# EmJEm on JVS (Colab runner)

All logic lives in the repository (`experiments/anatomy-inversion/run_jvs.sh`); this notebook only sets up a runtime and runs it.
Results and checkpoints go to Google Drive (`MyDrive/kanon-anatomy-inversion`), so a disconnected runtime can resume by rerunning all cells.

In [ ]:
BRANCH = "claude/voice-articulation-parameter-estimation-89f316"
MODES = "sp fix"          # EmJEm conditions to run, in order
ITERATIONS = 120
PER_ITERATION = 600       # utterances synthesized per iteration (CPU-bound; lower on small runtimes)
JVS_DRIVE_ID = "19oAw8wWn3Y7z6CKChRdAyGOB9yupL_Xt"  # official JVS zip (3.5 GB), see the JVS corpus page
import os; print(os.cpu_count(), "CPUs")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/kanon-anatomy-inversion"
os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%%bash -s "$BRANCH"
set -e
cd /content
[ -d kanon-vocal-lab-2 ] || git clone -q --depth 1 -b "$1" https://github.com/k0tokay/kanon-vocal-lab-2.git
cd kanon-vocal-lab-2 && git pull -q
apt-get -qq install -y libopenal-dev > /dev/null
cmake -S . -B build-linux -DVTL_BUILD_GUI=OFF -DCMAKE_BUILD_TYPE=Release > /dev/null
cmake --build build-linux -j"$(nproc)" 2>&1 | tail -1

In [ ]:
# Copy the official JVS zip into your Drive folder on the server side. Downloading the public file
# directly (gdown) often fails with "too many users have viewed or downloaded this file".
import time
ZIP = f"{DRIVE}/jvs_ver1.zip"
if not os.path.exists("/content/jvs_ver1") and not os.path.exists(ZIP):
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    service = build("drive", "v3")
    folders = service.files().list(q="name = 'kanon-anatomy-inversion' and 'root' in parents and "
                                     "mimeType = 'application/vnd.google-apps.folder' and trashed = false",
                                   fields="files(id)").execute()["files"]
    service.files().copy(fileId=JVS_DRIVE_ID, body={"name": "jvs_ver1.zip", "parents": [folders[0]["id"]]},
                         fields="id,size").execute()
    for _ in range(60):  # wait until the mounted Drive shows the copy
        if os.path.exists(ZIP):
            break
        time.sleep(10)
print(os.path.exists(ZIP) and os.path.getsize(ZIP))

In [ ]:
%%bash -s "$DRIVE"
set -e
cd /content
[ -d jvs_ver1 ] || unzip -q "$1/jvs_ver1.zip"
ls jvs_ver1 | head -3; ls jvs_ver1 | wc -l

In [ ]:
%%bash -s "$DRIVE"
cd /content/kanon-vocal-lab-2/experiments/anatomy-inversion
python3 check_api.py 2>&1 | grep -v "Creating\|ATTENTION" | tail -5

In [ ]:
%%bash -s "$DRIVE" "$MODES" "$ITERATIONS" "$PER_ITERATION"
set -e
cd /content/kanon-vocal-lab-2/experiments/anatomy-inversion
# The prepared data is cached on Drive and copied to local disk for speed.
mkdir -p data "$1/data"
if [ -f "$1/data/jvs-v1/meta.json" ]; then cp -rn "$1/data/jvs-v1" data/; fi
if [ ! -f data/jvs-v1/meta.json ]; then
  python3 prep_jvs.py /content/jvs_ver1 data/jvs-v1 | tail -2 && cp -r data/jvs-v1 "$1/data/"
fi
MODES="$2" ITERATIONS="$3" PER_ITERATION="$4" ./run_jvs.sh /content/jvs_ver1 data/jvs-v1 "$1/runs"
